# aule — Logging and progress bars

This notebook demonstrates aule's two observability features:

- **Structured logging** via the standard `logging` module with coloured, level-aware output — useful for debugging, understanding internal decisions (shape promotion, guardrail checks), and monitoring long pipelines.
- **Progress bars** via `tqdm` on computationally heavy loops (SSIM, gradient error, PSD, report generation) — shown automatically when `tqdm` is installed and the number of iterations exceeds a small threshold.

Both are opt-in and leave existing code completely unchanged.

In [1]:
!pip install -q aule
!pip install -q "aule[progress]"

## Logging

aule has a single global logger called `"aule"`. By default it is silent (level `WARNING`). Enable it with `aule.set_log_level()` — once per session, no per-call parameter needed.

In [2]:
import aule

# default: silent — nothing prints
import numpy as np
from aule.metrics import rmse
gt   = np.random.rand(8, 32, 32, 1)
pred = gt + 0.1 * np.random.randn(*gt.shape)
_ = rmse(gt, pred)   # silent
print('default level: no output above')

default level: no output above


In [3]:
# switch to INFO: see high-level operation notes
aule.set_log_level('INFO')

_ = rmse(gt, pred)
print('done')

done


In [4]:
# switch to DEBUG: see internal steps, shape decisions, force=True warnings etc.
aule.set_log_level('DEBUG')

from aule._shapes import to_canonical
import numpy as np

# canonicalising a series input logs the promotion
series = np.random.randn(4, 3, 100)
c = to_canonical(series, axes='bct')
print('canonical shape:', c.shape)

[aule] 21:18:32 DEBUG    Log level set to DEBUG
[aule] 21:18:32 DEBUG    Promoting series input with axes='bct' and shape (4, 3, 100) to canonical form


canonical shape: (4, 1, 1, 3, 100)


In [5]:
# guardrail force=True logs a WARNING even when level is DEBUG
import warnings
from aule.metrics import ssim

degenerate = np.random.rand(1, 1, 4)   # H=W=1
with warnings.catch_warnings(record=True):
    warnings.simplefilter('always')
    _ = ssim(degenerate, degenerate, force=True)  # WARNING logged to stderr

[aule] 21:18:32 WARNING  ssim called with force=True on a degenerate input (1, 1, 4)
[aule] 21:18:32 DEBUG    ssim: computing over 4 slices (B=1, C=4, T=1)


In [6]:
# restore quiet
aule.set_log_level('WARNING')
print('logger back to WARNING')

logger back to WARNING


### Environment variable alternative

You can also set the level before importing aule, which is useful in scripts and CI:

```bash
AULE_LOG_LEVEL=DEBUG python my_script.py
```

The env var is read once at import time; subsequent `set_log_level()` calls override it.

## Progress bars with tqdm

Three metrics use explicit Python loops over batch × channel × time slices and are slow on large inputs: `ssim`, `gradient_error`, `psd_radial_error`. When `tqdm` is installed and the batch size exceeds a small threshold (currently 4), a progress bar appears automatically — no extra argument needed.

In [7]:
# generate a reasonably large batch so the bar is visible
B, H, W, C = 8, 64, 64, 3
gt_big   = np.random.rand(B, H, W, C)
pred_big = gt_big + 0.05 * np.random.randn(B, H, W, C)

print('Input shape:', gt_big.shape, '→', B*C, 'slices to process')

Input shape: (8, 64, 64, 3) → 24 slices to process


In [8]:
from aule.metrics import ssim

score = ssim(gt_big, pred_big)
print(f'SSIM: {score:.4f}')

SSIM: 0.9848


In [9]:
from aule.metrics import gradient_error

score = gradient_error(gt_big, pred_big)
print(f'gradient_error: {score:.6f}')

gradient_error: 0.138835


In [10]:
from aule.metrics import psd_radial_error

score = psd_radial_error(gt_big, pred_big)
print(f'psd_radial_error: {score:.6f}')

psd_radial_error: 0.000114


### Progress bar in the validation report

`generate_report` calls ~20 metrics and ~9 plots in sequence. When `tqdm` is installed, two progress bars appear — one for the metrics loop and one for the plots loop — giving you a live estimate of how long the report will take.

In [11]:
from aule.report import generate_report
import os, tempfile

with tempfile.NamedTemporaryFile(suffix='.html', delete=False) as f:
    path = f.name

generate_report(gt_big, pred_big, save_path=path)
size_kb = os.path.getsize(path) / 1024
print(f'Report saved: {size_kb:.0f} KB')

report: plots: 100%|██████████| 8/8 [00:00<00:00, 14.95it/s]

Report saved: 679 KB


### Progress on time series metrics

`lag_correlation` and `dtw_distance` also show a progress bar when iterating over many batch members.

In [12]:
from aule.metrics import lag_correlation, dtw_distance

B, C, T = 8, 4, 200
ts_gt   = np.random.randn(B, C, T)
ts_pred = np.roll(ts_gt, 3, axis=-1) + 0.1 * np.random.randn(B, C, T)

print('lag_correlation:')
corr = lag_correlation(ts_gt, ts_pred, max_lag=30, axes='bct')
print(f'  peak lag: {np.arange(-30, 31)[np.argmax(corr)]}')

print('dtw_distance:')
dist = dtw_distance(ts_gt, ts_pred, axes='bct', window=20)
print(f'  DTW distance: {dist:.4f}')

lag_correlation:


  peak lag: 3
dtw_distance:


  DTW distance: 21.5802


### What happens without tqdm

If `tqdm` is not installed, aule falls back to plain loops silently — no error, no import warning, no change in output. The only difference is the absence of the progress bar.

```python
# no tqdm installed → same result, no bar
from aule.metrics import ssim
score = ssim(gt_big, pred_big)   # runs silently
```

## Using logging and tqdm together

They compose naturally: enable DEBUG logging to see each slice being processed, while tqdm shows overall progress.

In [13]:
aule.set_log_level('DEBUG')

# small batch so the output stays readable
small_gt   = np.random.rand(6, 16, 16, 1)
small_pred = small_gt + 0.1 * np.random.randn(*small_gt.shape)

score = ssim(small_gt, small_pred)
print(f'SSIM: {score:.4f}')

aule.set_log_level('WARNING')   # restore quiet

[aule] 21:18:37 DEBUG    Log level set to DEBUG
[aule] 21:18:37 DEBUG    ssim: computing over 6 slices (B=6, C=1, T=1)
                                                   

SSIM: 0.9438
